# Task 2: Class-wise Clustering Templates (M = 64)

This notebook clusters the 6,000 MNIST training vectors of each class into `M = 64` templates using `sklearn.cluster.KMeans`.

In [25]:
import numpy as np
import scipy.io as sio
from sklearn.cluster import KMeans
import time

In [ ]:
data = sio.loadmat('data_all.mat')

train_images = data['trainv'].astype(np.float64) / 255.0
train_labels = data['trainlab'].flatten().astype(int)
test_images = data['testv'].astype(np.float64) / 255.0
test_labels = data['testlab'].flatten().astype(int)

unique_labels, counts = np.unique(train_labels, return_counts=True)
print(f"train_images: {train_images.shape}, test_images: {test_images.shape}")
print(f"class counts: {dict(zip(unique_labels.tolist(), counts.tolist()))}")

train_images: (60000, 784), test_images: (10000, 784)
class counts: {0: 5923, 1: 6742, 2: 5958, 3: 6131, 4: 5842, 5: 5421, 6: 5918, 7: 6265, 8: 5851, 9: 5949}


In [27]:
M = 64
random_state = 42

class_templates = {}
class_cluster_ids = {}

for digit in range(10):
    train_vectors_class = train_images[train_labels == digit]

    kmeans = KMeans(n_clusters=M, random_state=random_state, n_init=10)
    cluster_ids = kmeans.fit_predict(train_vectors_class)
    templates = kmeans.cluster_centers_

    class_cluster_ids[digit] = cluster_ids
    class_templates[digit] = templates

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Library/Frameworks/Python.framework/Versions/3.12/lib/p

In [28]:
all_templates = np.vstack([class_templates[digit] for digit in range(10)])
all_template_labels = np.concatenate([np.full(M, digit, dtype=int) for digit in range(10)])

print(f"all_templates shape: {all_templates.shape}")
print(f"all_template_labels shape: {all_template_labels.shape}")

all_templates shape: (640, 784)
all_template_labels shape: (640,)


## 3. NN Evaluation With Cluster Templates vs Full Training Set

This section computes:
- confusion matrix and error rate for NN with `64 x 10 = 640` templates
- confusion matrix and error rate for NN with all 60,000 training vectors
- processing-time comparison and a short performance comment

In [29]:
def nn_predict_chunked(test_vectors, template_vectors, template_labels, chunk_size=2000):
    num_test = test_vectors.shape[0]
    num_templates = template_vectors.shape[0]

    test_square = np.sum(test_vectors ** 2, axis=1)
    best_distance = np.full(num_test, np.inf)
    best_label = np.zeros(num_test, dtype=template_labels.dtype)

    num_chunks = int(np.ceil(num_templates / chunk_size))
    for chunk_index in range(num_chunks):
        start = chunk_index * chunk_size
        end = min(start + chunk_size, num_templates)

        chunk_vectors = template_vectors[start:end]
        chunk_labels = template_labels[start:end]
        chunk_square = np.sum(chunk_vectors ** 2, axis=1)

        squared_distances = (
            test_square[:, None]
            + chunk_square[None, :]
            - 2.0 * test_vectors @ chunk_vectors.T
        )
        np.clip(squared_distances, 0, None, out=squared_distances)

        chunk_best_index = np.argmin(squared_distances, axis=1)
        chunk_best_distance = squared_distances[np.arange(num_test), chunk_best_index]

        improved = chunk_best_distance < best_distance
        best_distance[improved] = chunk_best_distance[improved]
        best_label[improved] = chunk_labels[chunk_best_index[improved]]

    return best_label


def confusion_matrix_and_error(y_true, y_pred, num_classes=10):
    matrix = np.zeros((num_classes, num_classes), dtype=int)
    for true_label, predicted_label in zip(y_true, y_pred):
        matrix[int(true_label), int(predicted_label)] += 1
    error_rate = 100.0 * np.mean(y_true != y_pred)
    return matrix, error_rate

In [30]:
start_time = time.perf_counter()
template_predictions = nn_predict_chunked(
    test_images,
    all_templates,
    all_template_labels,
    chunk_size=640,
)
template_time = time.perf_counter() - start_time

template_confusion_matrix, template_error_rate = confusion_matrix_and_error(
    test_labels,
    template_predictions,
)

print("Template NN results (640 templates)")
print(f"Error rate: {template_error_rate:.2f}%")
print(f"Processing time: {template_time:.3f} s")
print("Confusion matrix (rows=true, cols=pred):")
print(template_confusion_matrix)

Template NN results (640 templates)
Error rate: 4.66%
Processing time: 0.055 s
Confusion matrix (rows=true, cols=pred):
[[ 964    1    4    1    1    2    3    1    1    2]
 [   0 1130    1    0    0    0    3    0    0    1]
 [   9    7  982    5    2    0    5   11   11    0]
 [   0    0    9  934    1   25    0    5   24   12]
 [   1    5    4    0  928    0    7    4    2   31]
 [   3    0    2   12    0  852    8    1    7    7]
 [   7    3    0    0    3    4  939    0    2    0]
 [   0   19    9    0    5    0    0  960    1   34]
 [   4    1    5   13    6   22    1    4  912    6]
 [   5    4    5   10   24    2    1   20    5  933]]


/var/folders/75/d_gxn6h16d56g2617dp42cq00000gn/T/ipykernel_71417/1807856833.py:21: RuntimeWarning: divide by zero encountered in matmul
  - 2.0 * test_vectors @ chunk_vectors.T
/var/folders/75/d_gxn6h16d56g2617dp42cq00000gn/T/ipykernel_71417/1807856833.py:21: RuntimeWarning: overflow encountered in matmul
  - 2.0 * test_vectors @ chunk_vectors.T
/var/folders/75/d_gxn6h16d56g2617dp42cq00000gn/T/ipykernel_71417/1807856833.py:21: RuntimeWarning: invalid value encountered in matmul
  - 2.0 * test_vectors @ chunk_vectors.T


In [31]:
start_time = time.perf_counter()
full_predictions = nn_predict_chunked(
    test_images,
    train_images,
    train_labels,
    chunk_size=1000,
)
full_time = time.perf_counter() - start_time

full_confusion_matrix, full_error_rate = confusion_matrix_and_error(
    test_labels,
    full_predictions,
)

print("\nFull-training NN results (60,000 templates)")
print(f"Error rate: {full_error_rate:.2f}%")
print(f"Processing time: {full_time:.3f} s")
print("Confusion matrix (rows=true, cols=pred):")
print(full_confusion_matrix)

/var/folders/75/d_gxn6h16d56g2617dp42cq00000gn/T/ipykernel_71417/1807856833.py:21: RuntimeWarning: divide by zero encountered in matmul
  - 2.0 * test_vectors @ chunk_vectors.T
/var/folders/75/d_gxn6h16d56g2617dp42cq00000gn/T/ipykernel_71417/1807856833.py:21: RuntimeWarning: overflow encountered in matmul
  - 2.0 * test_vectors @ chunk_vectors.T
/var/folders/75/d_gxn6h16d56g2617dp42cq00000gn/T/ipykernel_71417/1807856833.py:21: RuntimeWarning: invalid value encountered in matmul
  - 2.0 * test_vectors @ chunk_vectors.T



Full-training NN results (60,000 templates)
Error rate: 3.09%
Processing time: 4.069 s
Confusion matrix (rows=true, cols=pred):
[[ 973    1    1    0    0    1    3    1    0    0]
 [   0 1129    3    0    1    1    1    0    0    0]
 [   7    6  992    5    1    0    2   16    3    0]
 [   0    1    2  970    1   19    0    7    7    3]
 [   0    7    0    0  944    0    3    5    1   22]
 [   1    1    0   12    2  860    5    1    6    4]
 [   4    2    0    0    3    5  944    0    0    0]
 [   0   14    6    2    4    0    0  992    0   10]
 [   6    1    3   14    5   13    3    4  920    5]
 [   2    5    1    6   10    5    1   11    1  967]]


In [32]:
speedup = full_time / template_time
error_increase = template_error_rate - full_error_rate

print("\nComparison summary")
print("------------------")
print(f"Template NN is {speedup:.2f}x faster than full-training NN.")
print(f"Error-rate change (template - full): {error_increase:+.2f} percentage points.")

if error_increase <= 0:
    print("Comment: The clustered-template approach is both faster and at least as accurate on this run.")
else:
    print(
        "Comment: The clustered-template approach is much faster, "
        "with a modest loss in accuracy compared to using all training vectors."
    )


Comparison summary
------------------
Template NN is 73.61x faster than full-training NN.
Error-rate change (template - full): +1.57 percentage points.
Comment: The clustered-template approach is much faster, with a modest loss in accuracy compared to using all training vectors.


## 4. KNN Classifier (K = 7)

This section builds a chunked KNN classifier with `K = 7`, computes confusion matrix and error rate, and compares it to:
- Template NN (640 templates, 1-NN)
- Full-training NN (60,000 templates, 1-NN)

In [ ]:
def knn_predict_chunked(test_vectors, template_vectors, template_labels, k=7, chunk_size=1000, num_classes=10):
    num_test = test_vectors.shape[0]
    num_templates = template_vectors.shape[0]

    if k <= 0 or k > num_templates:
        raise ValueError(f"k must be in [1, {num_templates}], got {k}")

    test_square = np.sum(test_vectors ** 2, axis=1)

    best_distances = np.full((num_test, k), np.inf)
    best_labels = np.zeros((num_test, k), dtype=template_labels.dtype)

    num_chunks = int(np.ceil(num_templates / chunk_size))
    row_index = np.arange(num_test)[:, None]

    for chunk_index in range(num_chunks):
        start = chunk_index * chunk_size
        end = min(start + chunk_size, num_templates)

        chunk_vectors = template_vectors[start:end]
        chunk_labels = template_labels[start:end]
        chunk_square = np.sum(chunk_vectors ** 2, axis=1)

        squared_distances = (
            test_square[:, None]
            + chunk_square[None, :]
            - 2.0 * test_vectors @ chunk_vectors.T
        )
        np.clip(squared_distances, 0, None, out=squared_distances)

        chunk_label_matrix = np.broadcast_to(chunk_labels, squared_distances.shape)
        merged_distances = np.concatenate([best_distances, squared_distances], axis=1)
        merged_labels = np.concatenate([best_labels, chunk_label_matrix], axis=1)

        best_indices = np.argpartition(merged_distances, kth=k - 1, axis=1)[:, :k]
        best_distances = merged_distances[row_index, best_indices]
        best_labels = merged_labels[row_index, best_indices]

    vote_counts = np.zeros((num_test, num_classes), dtype=int)
    vote_distance_sums = np.zeros((num_test, num_classes), dtype=np.float64)

    for class_label in range(num_classes):
        class_mask = (best_labels == class_label)
        vote_counts[:, class_label] = np.sum(class_mask, axis=1)
        vote_distance_sums[:, class_label] = np.sum(
            np.where(class_mask, best_distances, 0.0),
            axis=1,
        )

    max_votes = np.max(vote_counts, axis=1, keepdims=True)
    tied_classes = (vote_counts == max_votes)
    tie_break_dist = np.where(tied_classes, vote_distance_sums, np.inf)
    predictions = np.argmin(tie_break_dist, axis=1).astype(template_labels.dtype)

    return predictions


K = 7
start_time = time.perf_counter()
knn_predictions = knn_predict_chunked(
    test_images,
    train_images,
    train_labels,
    k=K,
    chunk_size=1000,
    num_classes=10,
)
knn_time = time.perf_counter() - start_time

knn_confusion_matrix, knn_error_rate = confusion_matrix_and_error(test_labels, knn_predictions)

print(f"KNN results (K={K}, 60,000 templates)")
print(f"Error rate: {knn_error_rate:.2f}%")
print(f"Processing time: {knn_time:.3f} s")
print("Confusion matrix (rows=true, cols=pred):")
print(knn_confusion_matrix)

print("\nThree-system comparison")
print("-----------------------")
print(f"Template NN (1-NN, 640 templates): error={template_error_rate:.2f}%, time={template_time:.3f}s")
print(f"Full NN (1-NN, 60,000 templates): error={full_error_rate:.2f}%, time={full_time:.3f}s")
print(f"KNN (K=7, 60,000 templates): error={knn_error_rate:.2f}%, time={knn_time:.3f}s")

best_error = min(template_error_rate, full_error_rate, knn_error_rate)
if knn_error_rate == best_error:
    print("Comment: KNN (K=7) gives the lowest error on this run.")
else:
    print("Comment: KNN (K=7) trades more computation for a different accuracy profile than 1-NN.")

/var/folders/75/d_gxn6h16d56g2617dp42cq00000gn/T/ipykernel_71417/1428739941.py:29: RuntimeWarning: divide by zero encountered in matmul
  - 2.0 * test_vectors @ chunk_vectors.T
/var/folders/75/d_gxn6h16d56g2617dp42cq00000gn/T/ipykernel_71417/1428739941.py:29: RuntimeWarning: overflow encountered in matmul
  - 2.0 * test_vectors @ chunk_vectors.T
/var/folders/75/d_gxn6h16d56g2617dp42cq00000gn/T/ipykernel_71417/1428739941.py:29: RuntimeWarning: invalid value encountered in matmul
  - 2.0 * test_vectors @ chunk_vectors.T


KNN results (K=7, 60,000 templates)
Error rate: 3.00%
Processing time: 8.167 s
Confusion matrix (rows=true, cols=pred):
[[ 974    1    1    0    0    1    2    1    0    0]
 [   0 1133    2    0    0    0    0    0    0    0]
 [  11    7  988    2    1    0    2   17    4    0]
 [   0    3    2  973    1   13    1    8    5    4]
 [   1    6    0    0  945    0    5    2    1   22]
 [   5    0    0    6    2  867    5    1    2    4]
 [   6    3    0    0    3    2  944    0    0    0]
 [   0   24    3    0    1    0    0  990    0   10]
 [   6    3    4   12    6    8    3    5  921    6]
 [   5    6    3    5    8    4    1   10    2  965]]

Three-system comparison
-----------------------
Template NN (1-NN, 640 templates): error=4.66%, time=0.055s
Full NN (1-NN, 60,000 templates): error=3.09%, time=4.069s
KNN (K=7, 60,000 templates): error=3.00%, time=8.167s
Comment: KNN (K=7) gives the lowest error on this run.
